# 🩻 Medical Image Diagnosis: train on a free GPU
Run this notebook on **Kaggle** (Settings -> Accelerator -> GPU) or **Google Colab** (Runtime -> Change runtime type -> T4 GPU).

Dataset used: **Brain Tumor MRI** (4 classes). To use chest X-rays instead, swap in `paultimothymooney/chest-xray-pneumonia`.

In [ ]:
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Get the project code
Replace the URL with *your* GitHub repo (or upload the project folder manually).

In [ ]:
!git clone https://github.com/YOUR_USERNAME/medical-diagnosis.git
%cd medical-diagnosis
!pip -q install gradio scikit-learn

## 2. Get the dataset
**On Kaggle:** click *Add Input* and search for **Brain Tumor MRI Dataset** (by Masoud Nickparvar). It appears under `/kaggle/input/`.

**On Colab:** upload your `kaggle.json` (Kaggle -> Settings -> Create New Token), then run the second cell.

In [ ]:
# --- Kaggle notebook: just point to the input folder ---
import os
DATA = '/kaggle/input/brain-tumor-mri-dataset' if os.path.exists('/kaggle/input') else 'data'
print(DATA, os.path.exists(DATA))

In [ ]:
# --- Colab only: download with the Kaggle API ---
import os
if not os.path.exists('/kaggle/input'):
    from google.colab import files
    files.upload()   # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data --unzip
    DATA = 'data'

## 3. Train
About 10-20 minutes on a T4. The script picks the best epoch by validation accuracy, then reports results on the untouched test set.

In [ ]:
!python train.py --data $DATA --arch resnet50 --head_epochs 2 --epochs 8 --workers 2

## 4. Look at the results

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open('outputs/metrics.json')), indent=2))
for f in ['confusion_matrix', 'training_curves', 'gradcam_examples']:
    display(Image(f'outputs/{f}.png'))

## 5. Save your model
Download `models/best_model.pt` and `outputs/` to your computer (they are not committed to GitHub because of file size).

In [ ]:
import shutil
shutil.make_archive('results', 'zip', '.', 'outputs')
try:
    from google.colab import files
    files.download('models/best_model.pt'); files.download('results.zip')
except ImportError:
    print('On Kaggle: find models/best_model.pt and results.zip in the Output tab after saving the notebook.')

## 6. Try the demo (public link, works in the notebook)

In [ ]:
import app
app.demo.launch(share=True)